In [ ]:
# Import Library
from selenium import webdriver
from selenium.webdriver.edge.service import Service
from selenium.webdriver.edge.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time
import pandas as pd
import os

# ====== INISIALISASI OPTIONS ======
edge_options = Options()
edge_options.add_argument(r"user-data-dir=D:\\Tools\\selenium_edge_profile")
edge_options.add_argument("profile-directory=Stockbit")
edge_options.add_argument('--disable-blink-features=AutomationControlled')
edge_options.add_experimental_option("excludeSwitches", ["enable-automation"])
edge_options.add_experimental_option('useAutomationExtension', False)

# Inisialisasi WebDriver menggunakan driver di folder Driver/msedgedriver.exe
Path_Webdriver = "./Driver/msedgedriver.exe"
try:
    service = Service(executable_path=Path_Webdriver)
    driver = webdriver.Edge(service=service, options=edge_options)
except Exception:
    driver = webdriver.Edge(options=edge_options)

wait = WebDriverWait(driver, 15)
driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")

# Daftar Sektor Utama Stockbit
target_sector_url = {
    "Barang-Konsumen-Primer": "https://stockbit.com/sector/barang-konsumen-primer",
    "Kesehatan" : "https://stockbit.com/sector/kesehatan",
    "Keuangan": "https://stockbit.com/sector/keuangan",
    "Barang-Konsumen-NonPrimer": "https://stockbit.com/sector/barang-konsumen-non-primer",
    "Properti dan Real Estat": "https://stockbit.com/sector/properti-real-estat",
    "Perindustrian": "https://stockbit.com/sector/perindustrian",
    "Energi": "https://stockbit.com/sector/energi",
    "Barang-Baku": "https://stockbit.com/sector/barang-baku",    
    "Infrastruktur": "https://stockbit.com/sector/infrastruktur",    
    "Teknologi": "https://stockbit.com/sector/teknologi",
    "Transportasi dan Logistik": "https://stockbit.com/sector/transportasi-logistik"
}

output = "Data Mentah/Perusahaan.csv"
os.makedirs(os.path.dirname(output), exist_ok=True)

# ============================================================
# Fungsi 1: Tekan Dropdown Sektor/Subsektor & Scroll Menu
# ============================================================
def open_and_scroll_dropdown(driver):
    '''
    Menekan elemen dropdown di bagian atas halaman (misal: 'All Sector > Kesehatan...')
    agar menu pilihan terbuka, kemudian melakukan scroll di dalam container dropdown
    tersebut untuk memuat seluruh opsi.
    '''
    time.sleep(1.5)
    
    # 1. Cari & Klik Elemen Trigger Dropdown
    trigger = None
    selectors = [
        '//div[contains(@class, "ant-cascader")]',
        '//div[contains(@class, "ant-select")]',
        '//*[contains(text(), "All Sector")]',
        '//div[contains(@class, "breadcrumb")]',
        '//span[contains(@class, "anticon-down")]/..',
        '//svg[contains(@data-icon, "down")]/ancestor::div[1]'
    ]
    
    for sel in selectors:
        try:
            elems = driver.find_elements(By.XPATH, sel)
            for el in elems:
                if el.is_displayed():
                    txt = el.text or ""
                    cls = el.get_attribute("class") or ""
                    if "All Sector" in txt or "ant-cascader" in cls or "ant-select" in cls:
                        trigger = el
                        break
            if trigger:
                break
        except Exception:
            pass

    if trigger:
        try:
            driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", trigger)
            time.sleep(0.3)
            driver.execute_script("arguments[0].click();", trigger)
            time.sleep(1)
        except Exception:
            pass

    # 2. Cari Container Menu Dropdown yang Terbuka & Lakukan Scroll
    menu_selectors = [
        '//div[contains(@class, "ant-select-dropdown")]',
        '//div[contains(@class, "ant-cascader-dropdown")]',
        '//ul[contains(@class, "ant-cascader-menu")]',
        '//div[contains(@class, "ant-dropdown")]',
        '//div[contains(@class, "rc-virtual-list")]'
    ]
    
    dropdown_menu = None
    for m_sel in menu_selectors:
        try:
            menus = driver.find_elements(By.XPATH, m_sel)
            for m in menus:
                if m.is_displayed():
                    dropdown_menu = m
                    break
            if dropdown_menu:
                break
        except Exception:
            pass

    if dropdown_menu:
        for _ in range(6):
            try:
                driver.execute_script("arguments[0].scrollTop += 300;", dropdown_menu)
                time.sleep(0.4)
            except Exception:
                break

# ============================================================
# Fungsi 2: Ambil Link Kategori / Sub-Sektor Rekursif
# ============================================================
def get_all_leaf_sector_urls(driver, start_url):
    '''
    Menelusuri hierarki subsektor/industri secara terstruktur 
    sampai menemukan URL paling ujung (leaf level) tempat tabel emiten berada.
    '''
    visited = set()
    to_visit = [start_url]
    leaf_urls = set()

    while to_visit:
        curr_url = to_visit.pop(0)
        if curr_url in visited:
            continue
        visited.add(curr_url)
        
        driver.get(curr_url)
        time.sleep(2)
        
        # Cek apakah ada link /symbol/ di halaman ini (artinya sudah di tingkat emiten)
        symbol_links = driver.find_elements(By.XPATH, '//a[contains(@href, "/symbol/")]')
        # Filter link symbol beneran (bukan navigasi header)
        valid_symbols = [l for l in symbol_links if "/symbol/" in (l.get_attribute("href") or "") and len((l.text or "").strip()) <= 6 and (l.text or "").strip().isupper()]
        
        # Ambil sub-sektor links di tabel
        table_sector_links = driver.find_elements(By.XPATH, '//div[contains(@class, "ant-table")]//tbody//a[contains(@href, "/sector/")]')
        
        new_sector_urls = []
        for l in table_sector_links:
            href = l.get_attribute("href") or ""
            if href and href not in visited and href != curr_url:
                new_sector_urls.append(href)
                
        if new_sector_urls:
            # Masih ada sub-sektor turunan, masukkan ke antrean
            to_visit.extend(new_sector_urls)
        else:
            # Tidak ada sub-sektor turunan lagi, halaman ini adalah halaman emiten
            leaf_urls.add(curr_url)
            
    return list(leaf_urls)

# ============================================================
# Fungsi 3: Scraping Ticker Emiten Saham (Virtual Scroll)
# ============================================================
def scrape_companies_virtual_scroll(driver):
    collected = {}
    no_new_streak = 0
    MAX_NO_NEW = 4

    while True:
        # Hanya ambil link yang menuju /symbol/ (Ticker Emiten)
        symbol_links = driver.find_elements(
            By.XPATH, '//div[contains(@class, "ant-table")]//tbody//a[contains(@href, "/symbol/")]'
        )
        count_before = len(collected)

        for el in symbol_links:
            code = el.get_attribute("textContent").strip()
            if not code:
                code = (el.get_attribute("title") or "").strip() or el.text.strip()
            
            link = el.get_attribute("href") or ''
            
            # Ticker emiten Indonesia 4-5 karakter kapital (misal KLBF, BBCA, GOTO)
            if code and len(code) <= 6 and code.isupper() and code not in collected and "/symbol/" in link and "/chartbit" not in link:
                collected[code] = link

        new_items = len(collected) - count_before
        if new_items > 0:
            no_new_streak = 0
        else:
            no_new_streak += 1
            if no_new_streak >= MAX_NO_NEW:
                break

        if symbol_links:
            try:
                driver.execute_script(
                    "arguments[0].scrollIntoView({block: 'nearest', behavior: 'smooth'});",
                    symbol_links[-1]
                )
            except Exception:
                pass

        time.sleep(0.8)

    return collected

# ============================================================
# Main Loop (Membangun Dataset: Code, Sektor, Sub_Sektor, Ticker_YF, Link)
# ============================================================
dataset = []

driver.get("https://stockbit.com/login")
time.sleep(3)

for sektor, url in target_sector_url.items():
    print(f"\\n{'='*50}")
    print(f"Processing Sektor Utama: {sektor}")
    print(f"URL: {url}")
    print(f"{'='*50}")
    
    # Dapatkan seluruh URL level terbawah (leaf) di dalam sektor ini
    leaf_urls = get_all_leaf_sector_urls(driver, url)
    print(f"  ✓ Ditemukan {len(leaf_urls)} sub-kategori terbawah di sektor {sektor}")
    
    for leaf_url in leaf_urls:
        # Ekstrak nama subsektor dari URL path
        sub_name = leaf_url.rstrip('/').split('/')[-1].replace('-', ' ').title()
        print(f"  --> Processing Halaman Emiten: {sub_name} ({leaf_url})")
        
        driver.get(leaf_url)
        time.sleep(2)
        
        # Buka dropdown & scroll lebih dulu jika ada virtual dropdown
        open_and_scroll_dropdown(driver)
        
        companies = scrape_companies_virtual_scroll(driver)
        print(f"      ✓ Terambil {len(companies)} emiten di {sub_name}")
        
        for code, link in companies.items():
            dataset.append({
                "Code": code,
                "Sektor": sektor,
                "Sub_Sektor": sub_name,
                "Ticker_YF": f"{code}.JK" if len(code) <= 5 and code.isalpha() else code,
                "Link": link
            })

# Simpan Ke CSV Dataset
df = pd.DataFrame(dataset)
if not df.empty:
    df = df.drop_duplicates(subset=["Code"]).reset_index(drop=True)
    df = df[["Code", "Sektor", "Sub_Sektor", "Ticker_YF", "Link"]]
    df.to_csv(output, index=False, encoding='utf-8-sig')

print(f"\\n{'='*60}")
print(f"✓ Dataset Berhasil Tersimpan ke: {output}")
print(f"✓ Total baris data (Emiten Unik): {len(df)}")
print(f"{'='*60}")



\n==================================================
Processing Sektor Utama: Barang-Konsumen-Primer
URL: https://stockbit.com/sector/barang-konsumen-primer
  ✓ Ditemukan 11 sub-kategori terbawah di sektor Barang-Konsumen-Primer
  --> Processing Halaman Emiten: Daging Ayam Ikan (https://stockbit.com/sector/barang-konsumen-primer/makanan-minuman/produk-makanan-pertanian/daging-ayam-ikan)
      ✓ Terambil 21 emiten di Daging Ayam Ikan
  --> Processing Halaman Emiten: Ritel Distributor Obat Obatan (https://stockbit.com/sector/barang-konsumen-primer/perdagangan-ritel-barang-primer/perdagangan-ritel-barang-primer/ritel-distributor-obat-obatan)
      ✓ Terambil 3 emiten di Ritel Distributor Obat Obatan
  --> Processing Halaman Emiten: Alkohol (https://stockbit.com/sector/barang-konsumen-primer/makanan-minuman/minuman/alkohol)
      ✓ Terambil 5 emiten di Alkohol
  --> Processing Halaman Emiten: Perawatan Tubuh (https://stockbit.com/sector/barang-konsumen-primer/produk-rumah-tangga-tidak-taha